# FlowEdit CFG-Like Interpolation Same-NFE Experiment

Goal: compare the original FlowEdit Euler update with a CFG-like FlowEdit interpolation update under the same estimated number of function evaluations (NFE).

The modification follows the first meeting idea: rewrite the FlowEdit editing field in a CFG-like contrastive form before applying the midpoint approximation scheme.

$$z_t^{src}=(1-t)x_{src}+t\epsilon$$

$$z_t^{tar}=z_t^{edit}+z_t^{src}-x_{src}$$

$$G_t^{FE}=v_{tar}(z_t^{tar},t)+[-v_{src}(z_t^{src},t)]$$

Then predict target/source midpoints separately:

$$\tilde z_{mid}^{tar}=z_t^{tar}+\frac{h}{2}v_{tar}(z_t^{tar},t)$$

$$\tilde z_{mid}^{src}=z_t^{src}+\frac{h}{2}v_{src}(z_t^{src},t)$$

$$G_{mid}^{FE}=v_{tar}(\tilde z_{mid}^{tar},t_{mid})+[-v_{src}(\tilde z_{mid}^{src},t_{mid})]$$

CFG-like interpolation update:

$$\hat G_t^{FE}=(1-\alpha(t))G_t^{FE}+\alpha(t)G_{mid}^{FE},\quad \alpha(t)=\lambda(1-t)^\gamma$$

The edit latent is updated by:

$$z_{next}^{edit}=z_t^{edit}+h\hat G_t^{FE}$$

Evaluation focus: same NFE, CLIP target alignment, DINO source preservation, runtime, and a lightweight saturation/clipping artifact proxy.


In [ ]:
# 1) Configuration
# Do not hard-code tokens in this notebook. Paste one only at runtime if SD3 access requires it.
import os
from getpass import getpass

REPO_URL = "https://github.com/Jiaqi-Ye/FlowEdit.git"
BRANCH = "codex/flowedit-cfg-like-interpolation"
WORKDIR = "/content/FlowEdit"

EXP_YAML = "SD3_cfg_like_interpolate_same_nfe.yaml"
DATASET_YAML = "edits_midpoint_eval.yaml"

METRICS_DIR = "outputs/metrics"
QUALITY_PER_SAMPLE_CSV = f"{METRICS_DIR}/cfg_like_interpolate_clip_dino_per_sample.csv"
QUALITY_SUMMARY_CSV = f"{METRICS_DIR}/cfg_like_interpolate_clip_dino_summary.csv"
ARTIFACT_PER_SAMPLE_CSV = f"{METRICS_DIR}/cfg_like_interpolate_artifact_per_sample.csv"
ARTIFACT_SUMMARY_CSV = f"{METRICS_DIR}/cfg_like_interpolate_artifact_summary.csv"
COMPARISON_CSV = f"{METRICS_DIR}/cfg_like_interpolate_comparison_table.csv"

HF_TOKEN = os.environ.get("HF_TOKEN", "")
if not HF_TOKEN:
    HF_TOKEN = getpass("Hugging Face token, leave blank if not needed: ")

print("Experiment YAML:", EXP_YAML)
print("Dataset YAML:", DATASET_YAML)
print("Target branch:", BRANCH)


In [ ]:
# 2) Clone repository, install dependencies, and switch to workspace
import os
import subprocess
from getpass import getpass
from pathlib import Path

# Match the environment setup used by the midpoint-solver notebook.
# If this cell changes installed binary packages after imports have already happened,
# choose Runtime > Restart runtime once, then rerun from Cell 1.

if not Path(WORKDIR).exists():
    subprocess.run(["git", "clone", REPO_URL, WORKDIR], check=True)

os.chdir(WORKDIR)
subprocess.run(["git", "fetch", "origin"], check=False)
checkout = subprocess.run(["git", "checkout", BRANCH], text=True, capture_output=True)
if checkout.returncode != 0:
    print(checkout.stdout)
    print(checkout.stderr)
    raise RuntimeError("Could not checkout the branch. Push this branch first, or upload the notebook into the checked-out repo.")
subprocess.run(["git", "pull", "--ff-only", "origin", BRANCH], check=False)
subprocess.run(["git", "status", "--short", "--branch"], check=False)

subprocess.run([
    "pip", "install", "-q", "--upgrade", "--force-reinstall",
    "numpy==2.0.2",
    "pandas==2.2.2",
    "scikit-learn==1.6.1",
    "pillow>=10,<12",
    "protobuf>=5.29.1,<7",
], check=True)
subprocess.run([
    "pip", "install", "-q", "--upgrade",
    "plotly==5.24.1",
    "diffusers>=0.31.0",
    "transformers>=4.44.0",
    "accelerate>=0.33.0",
    "safetensors",
    "sentencepiece",
    "einops",
    "pyyaml",
    "huggingface_hub",
], check=True)

try:
    import numpy as np
    import pandas as pd
    import sklearn
    import PIL
    import google.protobuf
    import plotly
    import diffusers
    import transformers
    print("numpy", np.__version__)
    print("pandas", pd.__version__)
    print("scikit-learn", sklearn.__version__)
    print("Pillow", PIL.__version__)
    print("protobuf", google.protobuf.__version__)
    print("plotly", plotly.__version__)
    print("diffusers", diffusers.__version__)
    print("transformers", transformers.__version__)
except ValueError:
    print("Package ABI error detected. In Colab, choose Runtime > Restart runtime, then rerun from Cell 1.")
    raise

HF_TOKEN = globals().get("HF_TOKEN") or os.environ.get("HF_TOKEN", "")
if not HF_TOKEN:
    HF_TOKEN = getpass("Hugging Face token, leave blank if not needed: " )

if HF_TOKEN:
    from huggingface_hub import login
    login(token=HF_TOKEN, add_to_git_credential=False)
    print("Logged in to Hugging Face for model download.")
else:
    print("No HF token provided. Public downloads only.")


In [ ]:
# 3) Inspect the same-NFE experiment configuration
from pathlib import Path
import yaml
import pandas as pd

with open(EXP_YAML, "r", encoding="utf-8") as f:
    exp = yaml.safe_load(f)

two_call_edit_solvers = {
    "midpoint",
    "flowedit_cfg_like_interpolate",
    "flowedit_cfg_like_interp",
    "flowedit_cfg_interpolate",
    "flowedit_cfg_interp",
}

rows = []
for item in exp:
    edit_steps = max(min(item["n_max"], item["T_steps"]) - max(item["n_min"], 0), 0)
    calls_per_edit_step = 2 if item["solver_type"] in two_call_edit_solvers else 1
    estimated_nfe = edit_steps * item["n_avg"] * calls_per_edit_step
    rows.append({
        "exp_name": item["exp_name"],
        "solver_type": item["solver_type"],
        "T_steps": item["T_steps"],
        "n_max": item["n_max"],
        "n_avg": item["n_avg"],
        "estimated_nfe": estimated_nfe,
        "pc_lambda": item.get("pc_guidance_lambda", ""),
        "pc_gamma": item.get("pc_guidance_gamma", ""),
    })

display(pd.DataFrame(rows))
print(Path(EXP_YAML).read_text())


In [ ]:
# 4) Run original FlowEdit vs CFG-like FlowEdit interpolation under the same estimated NFE
import shutil
import subprocess
from pathlib import Path
import pandas as pd

cleanup_paths = [
    "outputs/SameNFE_OriginalFlowEdit_Euler18",
    "outputs/SameNFE_CFGLikeFlowEdit_Interpolate18_L050",
    "outputs/run_summary.csv",
    QUALITY_PER_SAMPLE_CSV,
    QUALITY_SUMMARY_CSV,
    ARTIFACT_PER_SAMPLE_CSV,
    ARTIFACT_SUMMARY_CSV,
    COMPARISON_CSV,
]
for path in cleanup_paths:
    p = Path(path)
    if p.is_dir():
        shutil.rmtree(p)
    elif p.exists():
        p.unlink()

Path(METRICS_DIR).mkdir(parents=True, exist_ok=True)
subprocess.run(["python", "run_script.py", "--device_number", "0", "--exp_yaml", EXP_YAML], check=True)

print("Run summary:")
display(pd.read_csv("outputs/run_summary.csv"))


In [ ]:
# 5) Evaluate generation quality and artifact proxy
import subprocess
from pathlib import Path
import pandas as pd

subprocess.run([
    "python", "evaluate_clip_dino.py",
    "--run_summary_csv", "outputs/run_summary.csv",
    "--dataset_yaml", DATASET_YAML,
    "--out_samples", QUALITY_PER_SAMPLE_CSV,
    "--out_summary", QUALITY_SUMMARY_CSV,
], check=True)

subprocess.run([
    "python", "evaluate_artifact_proxy.py",
    "--run_summary_csv", "outputs/run_summary.csv",
    "--out_samples", ARTIFACT_PER_SAMPLE_CSV,
    "--out_summary", ARTIFACT_SUMMARY_CSV,
], check=True)

quality = pd.read_csv(QUALITY_SUMMARY_CSV)
artifact = pd.read_csv(ARTIFACT_SUMMARY_CSV)
summary = quality.merge(
    artifact,
    on=["exp_name", "solver_type", "estimated_nfe", "pc_guidance_lambda", "pc_guidance_gamma", "num_samples"],
    how="left",
)
display(summary)
summary.to_csv(COMPARISON_CSV, index=False)
print("Wrote:", COMPARISON_CSV)

In [ ]:
# 6) Per-sample same-NFE comparison table
from pathlib import Path
import pandas as pd

quality_samples = pd.read_csv(QUALITY_PER_SAMPLE_CSV)
artifact_samples = pd.read_csv(ARTIFACT_PER_SAMPLE_CSV)
samples = quality_samples.merge(
    artifact_samples[[
        "exp_name", "solver_type", "source_image", "target_index",
        "edited_artifact_proxy", "artifact_proxy_delta_vs_source",
        "edited_clipping_ratio", "edited_high_saturation_ratio",
    ]],
    on=["exp_name", "solver_type", "source_image", "target_index"],
    how="left",
)
samples["case"] = samples["source_image"].map(lambda p: Path(p).stem)

metrics = [
    "clip_alignment",
    "dino_similarity",
    "edit_preservation_score",
    "edited_artifact_proxy",
    "artifact_proxy_delta_vs_source",
    "elapsed_seconds",
]
for col in metrics:
    samples[col] = samples[col].astype(float)

display(samples[["case", "solver_type", "estimated_nfe"] + metrics].sort_values(["case", "solver_type"]))

In [ ]:
# 7) Image comparison table for visual artifact inspection
import base64
import html
from pathlib import Path
import pandas as pd
import yaml
from IPython.display import HTML, display

quality_samples = pd.read_csv(QUALITY_PER_SAMPLE_CSV)
artifact_samples = pd.read_csv(ARTIFACT_PER_SAMPLE_CSV)
samples = quality_samples.merge(
    artifact_samples[[
        "exp_name", "solver_type", "source_image", "target_index",
        "edited_artifact_proxy", "artifact_proxy_delta_vs_source",
    ]],
    on=["exp_name", "solver_type", "source_image", "target_index"],
    how="left",
)
with open(DATASET_YAML, "r", encoding="utf-8") as f:
    dataset = yaml.safe_load(f)

solver_columns = [
    ("euler", "Original FlowEdit Euler"),
    ("flowedit_cfg_like_interpolate", "CFG-like FlowEdit interpolation"),
]

def image_to_data_uri(path):
    path = Path(path)
    suffix = path.suffix.lower().replace(".", "")
    if suffix == "jpg":
        suffix = "jpeg"
    data = base64.b64encode(path.read_bytes()).decode("ascii")
    return f"data:image/{suffix};base64,{data}"

def img_tag(path, width=220):
    return f"<img src='{image_to_data_uri(path)}' style='width:{width}px;max-width:100%;border:1px solid #ddd;'>"

def metric_line(row):
    return (
        f"CLIP {float(row['clip_alignment']):.4f} / "
        f"DINO {float(row['dino_similarity']):.4f} / "
        f"Artifact {float(row['edited_artifact_proxy']):.4f}"
    )

rows = []
for item in dataset:
    source = item["input_img"]
    subset = samples[samples["source_image"] == source]
    cells = [f"<td><b>{html.escape(Path(source).stem)}</b><br>{img_tag(source)}</td>"]
    for solver_type, label in solver_columns:
        solver_subset = subset[subset["solver_type"].str.lower() == solver_type]
        if solver_subset.empty:
            cells.append(f"<td><b>{html.escape(label)}</b><br><em>missing output</em></td>")
            continue
        row = solver_subset.iloc[0]
        cells.append(
            f"<td><b>{html.escape(label)}</b><br>{img_tag(row['output_image'])}<br>{metric_line(row)}</td>"
        )
    rows.append("<tr>" + "".join(cells) + "</tr>")

headers = "".join(f"<th>{html.escape(label)}</th>" for _, label in solver_columns)
table_html = (
    "<table style='border-collapse:collapse;width:100%;'>\n"
    f"<thead><tr><th>Source</th>{headers}</tr></thead>\n"
    "<tbody>\n"
    + "\n".join(rows)
    + "\n</tbody></table>"
)
display(HTML(table_html))


## How to Interpret

- Same NFE means the comparison controls for the number of neural-network flow evaluations.
- Higher CLIP alignment means better target-prompt matching.
- Higher DINO similarity means stronger source structure preservation.
- Lower artifact proxy and lower artifact delta suggest fewer saturation/clipping artifacts, but this is only a lightweight proxy; use the image table for final qualitative judgment.
- The CFG-like interpolation version tests whether rewriting FlowEdit as target velocity plus negative source velocity gives a cleaner midpoint approximation than the original Euler update.
- A convincing next claim would be: at the same estimated NFE, CFG-like FlowEdit interpolation improves target alignment or artifact behavior without collapsing source preservation.
